# Week 7 - Task 2: Model Selection & Hyperparameter Optimization

## Objective

This project compares Random Forest, Gradient Boosting, and Support Vector Machine (SVM) classifiers and uses both `GridSearchCV` and `RandomizedSearchCV` to optimize hyperparameters.

### Workflow
1. Load and inspect a tabular classification dataset.
2. Split data into training, validation, and test sets.
3. Establish baseline models.
4. Define documented hyperparameter search spaces.
5. Tune Random Forest with GridSearchCV.
6. Tune Gradient Boosting with RandomizedSearchCV.
7. Tune SVM with GridSearchCV.
8. Evaluate final models on the untouched test set.
9. Compare accuracy, precision, recall, F1, ROC-AUC, training/search time, and complexity.
10. Select a final model based on performance and practical trade-offs.

## Dataset

This project uses the **Breast Cancer Wisconsin (Diagnostic)** dataset available directly through Scikit-Learn.

It is a binary classification problem:
- Target 0: malignant
- Target 1: benign

The dataset has 569 observations and 30 numerical features.

Using a built-in dataset makes the project reproducible without requiring a separate download.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from scipy.stats import randint, uniform, loguniform

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Dataset shape:", X.shape)
display(X.head())
print("\nTarget distribution:")
display(y.value_counts().rename(index={0:"malignant", 1:"benign"}))

## 1. Train, Validation, and Test Sets

The assignment requires training, validation, and test sets.

We create:
- **64% training**
- **16% validation**
- **20% test**

The validation set is used for additional inspection while the cross-validation procedures perform model selection using the training data. The test set remains untouched until final evaluation.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 2. Baseline Models

Before tuning, we train three baseline algorithms:

- Random Forest
- Gradient Boosting
- SVM

SVM uses `StandardScaler` because its optimization is sensitive to feature scale. Tree-based models do not require feature scaling.

In [ ]:
baseline_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ])
}

def evaluate_model(model, X_eval, y_eval):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    pred = model.predict(X_eval)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_eval)[:, 1]
    else:
        prob = model.decision_function(X_eval)

    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred),
        "Recall": recall_score(y_eval, pred),
        "F1": f1_score(y_eval, pred),
        "ROC_AUC": roc_auc_score(y_eval, prob),
        "Fit_Time_sec": fit_time,
        "Predictions": pred
    }

baseline_results = []
for name, model in baseline_models.items():
    result = evaluate_model(model, X_val, y_val)
    result["Model"] = name
    baseline_results.append(result)

baseline_table = pd.DataFrame(baseline_results).drop(columns=["Predictions"]).set_index("Model")
display(baseline_table.round(4))

## 3. Hyperparameter Search Spaces

The search spaces below are chosen from commonly documented Scikit-Learn estimator parameters.

### Random Forest - GridSearchCV
We search:
- `n_estimators`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `max_features`

### Gradient Boosting - RandomizedSearchCV
We sample:
- `n_estimators`
- `learning_rate`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `subsample`

### SVM - GridSearchCV
We search:
- `C`
- `gamma`
- `kernel`

The scoring metric is F1 because the task is binary classification and F1 balances precision and recall.

In [ ]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_grid = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_search = GridSearchCV(
    rf,
    param_grid=rf_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)

gb_distributions = {
    "n_estimators": randint(50, 401),
    "learning_rate": loguniform(0.01, 0.3),
    "max_depth": randint(1, 6),
    "min_samples_split": randint(2, 11),
    "min_samples_leaf": randint(1, 6),
    "subsample": uniform(0.7, 0.3)
}

gb_search = RandomizedSearchCV(
    gb,
    param_distributions=gb_distributions,
    n_iter=30,
    scoring="f1",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True, random_state=RANDOM_STATE))
])

svm_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    "model__kernel": ["rbf", "linear"]
}

svm_search = GridSearchCV(
    svm_pipeline,
    param_grid=svm_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

## 4. Run Hyperparameter Optimization

GridSearchCV exhaustively evaluates every combination in its grid. RandomizedSearchCV samples a fixed number of combinations from the specified distributions.

Search time is recorded because training time is part of the final model-selection justification.

In [ ]:
search_objects = [
    ("Random Forest", rf_search),
    ("Gradient Boosting", gb_search),
    ("SVM", svm_search)
]

search_results = []
for name, search in search_objects:
    start = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    search_results.append({
        "Model": name,
        "Best_CV_F1": search.best_score_,
        "Search_Time_sec": elapsed,
        "Best_Params": search.best_params_,
        "Best_Estimator": search.best_estimator_
    })

    print(f"{name}")
    print("Best CV F1:", round(search.best_score_, 4))
    print("Search time:", round(elapsed, 2), "sec")
    print("Best params:", search.best_params_)
    print()

## 5. Validation-Set Comparison

The tuned estimators are evaluated on the held-out validation set before the final test evaluation.

In [ ]:
tuned_validation = []

for row in search_results:
    model = row["Best_Estimator"]
    pred = model.predict(X_val)
    prob = model.predict_proba(X_val)[:, 1]

    tuned_validation.append({
        "Model": row["Model"],
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
        "ROC_AUC": roc_auc_score(y_val, prob),
        "Search_Time_sec": row["Search_Time_sec"]
    })

tuned_validation_table = pd.DataFrame(tuned_validation).set_index("Model")
display(tuned_validation_table.round(4))

## 6. Final Test-Set Evaluation

After hyperparameter selection, the tuned models are evaluated on the untouched test set.

This is the most important performance estimate because the test set was not used to choose hyperparameters.

In [ ]:
final_results = []

for row in search_results:
    model = row["Best_Estimator"]
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    final_results.append({
        "Model": row["Model"],
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "Search_Time_sec": row["Search_Time_sec"],
        "Complexity": str(model)
    })

final_table = pd.DataFrame(final_results).set_index("Model")
display(final_table.drop(columns=["Complexity"]).round(4))

## 7. Comparative Analysis

We compare the tuned models using:
- F1 and ROC-AUC for predictive performance.
- Accuracy, precision, and recall for interpretability of classification behavior.
- Search time as a practical resource consideration.
- Model complexity as a deployment consideration.

The final choice should not automatically be the model with the highest score. A small performance advantage may not justify substantially higher training time or complexity.

In [ ]:
comparison = final_table.drop(columns=["Complexity"]).copy()

# Simple ranking: prioritize F1, then ROC-AUC, then shorter search time.
comparison["F1_Rank"] = comparison["F1"].rank(ascending=False, method="min")
comparison["ROC_AUC_Rank"] = comparison["ROC_AUC"].rank(ascending=False, method="min")
comparison["Time_Rank"] = comparison["Search_Time_sec"].rank(ascending=True, method="min")
comparison["Overall_Rank_Score"] = (
    comparison["F1_Rank"] + comparison["ROC_AUC_Rank"] + comparison["Time_Rank"]
)

display(comparison.sort_values("Overall_Rank_Score").round(4))

## 8. Confusion Matrix of the Best Model

The best model according to the ranking above is inspected using a confusion matrix.

In [ ]:
best_name = comparison.sort_values("Overall_Rank_Score").index[0]
best_model = next(row["Best_Estimator"] for row in search_results if row["Model"] == best_name)

best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)
ax.set_title(f"Confusion Matrix - {best_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.savefig("../docs/best_model_confusion_matrix.png", dpi=150)
plt.show()

print("Selected model:", best_name)
print(classification_report(y_test, best_pred, target_names=dataset.target_names))

## 9. Final Model Selection

The final model is selected using the measured test performance, validation behavior, optimization cost, and model complexity.

A practical selection should favor a model with strong F1/ROC-AUC while avoiding unnecessary search or inference complexity.

The exact measured values are written to the generated comparison files after the notebook is executed.

In [ ]:
os.makedirs("../docs", exist_ok=True)

final_table.drop(columns=["Complexity"]).to_csv("../docs/final_model_comparison.csv")

with open("../docs/best_model.txt", "w", encoding="utf-8") as f:
    f.write(f"Selected model: {best_name}\n")
    f.write(f"Reason: best combined ranking of F1, ROC-AUC, and search time.\n")

print("Saved final comparison and best-model summary.")